### DELTALAKE FEATURES

### Sample Dataset

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Delta_Project").getOrCreate()
data = [
    (1, "C001", "Laptop", 50000),
    (2, "C002", "Mobile", 15000),
    (3, "C003", "Tablet", 20000),
    (4, "C004", "Laptop", 55000)
]
columns = ["id", "customer_id", "product", "amount"]
df = spark.createDataFrame(data, columns)

In [0]:
df.display()

id,customer_id,product,amount
1,C001,Laptop,50000
2,C002,Mobile,15000
3,C003,Tablet,20000
4,C004,Laptop,55000


### Creating an Delta Table

In [0]:
path='/Volumes/workspace/default/deltalakeassignment'
df.write.format('delta').mode('overwrite').save(path)

### Insert Data

In [0]:
new_data=[(5, "C005", "Camera", 30000)]
new_df=spark.createDataFrame(new_data, columns)
new_df.write.format('delta').mode('append').save(path)

### Update Data

In [0]:
from delta.tables import DeltaTable
delta_table=DeltaTable.forPath(spark,path)
delta_table.update(
    condition='id=2',
    set={'amount':'18000'}
)

DataFrame[num_affected_rows: bigint]

### Delete Data

In [0]:
delta_table.delete('id=1')

DataFrame[num_affected_rows: bigint]

### Merge Data

In [0]:
updates = [
    (3, "C003", "Tablet", 22000),
    (6, "C006", "Watch", 8000)
]
updates_df=spark.createDataFrame(updates, columns)
delta_table.alias('target').merge(updates_df.alias('source'),'target.id=source.id').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(spark.sql(f"DESCRIBE HISTORY delta.`{path}`"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
15,2026-04-17T05:42:41.000Z,77333947973879,22pa1a4506@vishnu.edu.in,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(350219182475234),9c4e82b5-6648-469c-b905-11929f8964d7,0417-054120-fmeuua5z-v2n,14,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3807, p25FileSize -> 1353, numDeletionVectorsRemoved -> 1, minFileSize -> 1353, numAddedFiles -> 1, maxFileSize -> 1353, p75FileSize -> 1353, p50FileSize -> 1353, numAddedBytes -> 1353)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
14,2026-04-17T05:42:38.000Z,77333947973879,22pa1a4506@vishnu.edu.in,MERGE,"Map(predicate -> [""(id#11803L = id#11823L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(350219182475234),9c4e82b5-6648-469c-b905-11929f8964d7,0417-054120-fmeuua5z-v2n,12,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 2475, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 5119, materializeSourceTimeMs -> 546, numTargetRowsInserted -> 1, conflictDetectionTimeMs -> 403, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 1, scanTimeMs -> 1991, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 1, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2440)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
13,2026-04-17T05:42:32.000Z,77333947973879,22pa1a4506@vishnu.edu.in,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(350219182475234),3014fc9f-be07-49c2-84a4-82c87a4b9f04,0417-054120-fmeuua5z-v2n,12,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1374, p25FileSize -> 1332, numDeletionVectorsRemoved -> 1, minFileSize -> 1332, numAddedFiles -> 1, maxFileSize -> 1332, p75FileSize -> 1332, p50FileSize -> 1332, numAddedBytes -> 1332)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
12,2026-04-17T05:42:30.000Z,77333947973879,22pa1a4506@vishnu.edu.in,DELETE,"Map(predicate -> [""(id#11487L = 1)""])",null,List(350219182475234),3014fc9f-be07-49c2-84a4-82c87a4b9f04,0417-054120-fmeuua5z-v2n,10,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 2275, conflictDetectionTimeMs -> 740, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 1542, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 727)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
11,2026-04-17T05:42:28.000Z,77333947973879,22pa1a4506@vishnu.edu.in,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(350219182475234),46840041-113c-47a2-95a6-c89d9fbae275,0417-054120-fmeuua5z-v2n,10,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3828, p25FileSize -> 1374, numDeletionVectorsRemoved -> 1, minFileSize -> 1374, numAddedFiles -> 1, maxFileSize -> 1374, p75FileSize -> 1374, p50FileSize -> 1374, numAddedBytes -> 1374)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
10,2026-04-17T05:42:24.000Z,77333947973879,22pa1a4506@vishnu.edu.in,UPDATE,"Map(predicate -> [""(id#11164L = 2)""])",null,List(350219182475234),46840041-113c-47a2-95a6-c89d9fbae275,0417-054120-fmeuua5z-v2n,9,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRow

### Schema Evolution

In [0]:
from pyspark.sql.functions import lit
df_with_country=spark.read.format('delta').load(path).withColumn('country',lit('India'))

In [0]:
df_with_country.write.format('delta').mode('overwrite').option('mergeSchema','true').save(path)

### Time Travel

### View History

In [0]:
spark.sql(f"DESCRIBE HISTORY delta.`{path}`").display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
16,2026-04-17T05:51:31.000Z,77333947973879,22pa1a4506@vishnu.edu.in,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(350219182475234),e202cbae-e83b-41e5-8d91-e55978c5d7cd,0417-054120-fmeuua5z-v2n,15,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1353, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1572)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
15,2026-04-17T05:42:41.000Z,77333947973879,22pa1a4506@vishnu.edu.in,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(350219182475234),9c4e82b5-6648-469c-b905-11929f8964d7,0417-054120-fmeuua5z-v2n,14,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3807, p25FileSize -> 1353, numDeletionVectorsRemoved -> 1, minFileSize -> 1353, numAddedFiles -> 1, maxFileSize -> 1353, p75FileSize -> 1353, p50FileSize -> 1353, numAddedBytes -> 1353)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
14,2026-04-17T05:42:38.000Z,77333947973879,22pa1a4506@vishnu.edu.in,MERGE,"Map(predicate -> [""(id#11803L = id#11823L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(350219182475234),9c4e82b5-6648-469c-b905-11929f8964d7,0417-054120-fmeuua5z-v2n,12,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 2475, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 5119, materializeSourceTimeMs -> 546, numTargetRowsInserted -> 1, conflictDetectionTimeMs -> 403, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 1, scanTimeMs -> 1991, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 1, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2440)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
13,2026-04-17T05:42:32.000Z,77333947973879,22pa1a4506@vishnu.edu.in,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(350219182475234),3014fc9f-be07-49c2-84a4-82c87a4b9f04,0417-054120-fmeuua5z-v2n,12,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1374, p25FileSize -> 1332, numDeletionVectorsRemoved -> 1, minFileSize -> 1332, numAddedFiles -> 1, maxFileSize -> 1332, p75FileSize -> 1332, p50FileSize -> 1332, numAddedBytes -> 1332)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
12,2026-04-17T05:42:30.000Z,77333947973879,22pa1a4506@vishnu.edu.in,DELETE,"Map(predicate -> [""(id#11487L = 1)""])",null,List(350219182475234),3014fc9f-be07-49c2-84a4-82c87a4b9f04,0417-054120-fmeuua5z-v2n,10,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 2275, conflictDetectionTimeMs -> 740, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 1542, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 727)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
11,2026-04-17T05:42:28.000Z,77333947973879,22pa1a4506@vishnu.edu.in,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(350219182475234),46840041-113c-47a2-95a6-c89d9fbae275,0417-054120-fmeuua5z-v2n,10,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3828, p25FileSize -> 1374, numDeletionVectorsRemoved -> 1, minFileSize -> 1374, 

### To read old data

In [0]:
old_df=spark.read.format('delta')\
    .option('versionAsOf',0).load(path)
### Insert
old_df_1=spark.read.format('delta')\
    .option('versionAsOf',1).load(path)
### Update
old_df_2=spark.read.format('delta')\
    .option('versionAsOf',2).load(path)
### Merge
old_df_3=spark.read.format('delta')\
    .option('versionAsOf',3).load(path)
### Delete
old_df_4=spark.read.format('delta')\
    .option('versionAsOf',4).load(path)

In [0]:
old_df.display()
old_df_1.display()
old_df_2.display()
old_df_3.display()
old_df_4.display()

id,customer_id,product,amount
1,C001,Laptop,50000
2,C002,Mobile,15000
3,C003,Tablet,20000
4,C004,Laptop,55000


id,customer_id,product,amount
1,C001,Laptop,50000
2,C002,Mobile,15000
3,C003,Tablet,20000
4,C004,Laptop,55000
5,C005,Camera,30000


id,customer_id,product,amount
1,C001,Laptop,50000
3,C003,Tablet,20000
4,C004,Laptop,55000
5,C005,Camera,30000
2,C002,Mobile,18000


id,customer_id,product,amount
2,C002,Mobile,18000
1,C001,Laptop,50000
3,C003,Tablet,20000
4,C004,Laptop,55000
5,C005,Camera,30000


id,customer_id,product,amount
2,C002,Mobile,18000
3,C003,Tablet,20000
4,C004,Laptop,55000
5,C005,Camera,30000


### Final Output validation

In [0]:
final_df = spark.read.format("delta").load(path)
final_df.show()
final_df.printSchema()

+---+-----------+-------+------+-------+
| id|customer_id|product|amount|country|
+---+-----------+-------+------+-------+
|  2|       C002| Mobile| 18000|  India|
|  4|       C004| Laptop| 55000|  India|
|  5|       C005| Camera| 30000|  India|
|  3|       C003| Tablet| 22000|  India|
|  6|       C006|  Watch|  8000|  India|
+---+-----------+-------+------+-------+

root
 |-- id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: long (nullable = true)
 |-- country: string (nullable = true)

